### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [1]:
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [2]:
## Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k":4, "lambda_mult":0.7})


d:\Mission Project\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6833.82it/s]


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

llm = ChatGroq(model= "openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000270E116AE40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000270D6F3C2F0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
## Step 3: Query Decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [5]:
query = "How does Langchain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question": query})

In [6]:
print(decomposition_question)

**Possible sub‑questions**

1. How does LangChain implement and manage memory (e.g., vector stores, conversation buffers, retrievers) for LLM‑driven applications?  
2. What types of agents does LangChain provide, and how are they structured (tools, planning, execution, callbacks) within the framework?  
3. How does CrewAI handle memory and state management for its autonomous agents?  
4. In what ways do LangChain’s memory and agent architectures differ from or resemble those used in CrewAI (e.g., design patterns, extensibility, integration with external tools)?


In [8]:
## Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [10]:
## Step 5: Full RAG Pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]

    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [11]:
## Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("Final Answer: \n")
print(final_answer)

Final Answer: 

Q: **Decomposed Sub‑questions**
A: Below are a set of concrete sub‑questions you can ask to explore each claim made in the provided context. They break the overall topic into focused, answerable pieces that together give a complete picture of the CrewAI framework.

| # | Sub‑question | Why it matters |
|---|--------------|----------------|
| 1 | **What are the “standard patterns” (Stuff, Map‑Reduce, Refine) that CrewAI supports, and how does each pattern work?** | Understanding the building blocks lets you know how to compose and reuse chains effectively. |
| 2 | **How does CrewAI’s “Stuff” pattern differ from the “Map‑Reduce” and “Refine” patterns in terms of data flow and agent responsibilities?** | Highlights the trade‑offs and appropriate use‑cases for each pattern. |
| 3 | **What does “full traceability of agent decisions and interactions” mean in practice?** | Clarifies what information is logged, how it is stored, and how it can be accessed. |
| 4 | **Which tools